In [1]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-03-27 09:22:34--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.04s   

2026-03-27 09:22:34 (28.2 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [ ]:
with open('input.txt', 'r', encoding='utf-8') as f: 
    text = f.read()

len(text)

1115394

In [ ]:
# create a set of all characters that appear, then turn it into a sorted list
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [4]:
# tokenize the text
# very simple mapping using the index of the character (sorted based on the ASCII value of each character)

stoi = {char : i for i, char in enumerate(chars)}
itos = {i : char for i, char in enumerate(chars)}

# print(stoi)
# print(itos)

# encode = lambda char : stoi[char]
# decode = lambda i : itos[i]
encode = lambda s : [stoi[char] for char in s] # take a string and turn it into a list of integers
decode = lambda l : ''.join([itos[i] for i in l]) # take a list of integers and turn it into a string

print(encode("wowowwowowhello!"))
print(decode(encode('wowowwowowhello')))

[61, 53, 61, 53, 61, 61, 53, 61, 53, 61, 46, 43, 50, 50, 53, 2]
wowowwowowhello


In [5]:
import torch 
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)

torch.Size([1115394]) torch.int64


In [6]:
# train/val split
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

train_data[:10]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47])

In [7]:
block_size = 8
x = train_data[:block_size]
y = train_data[1:block_size+1]

for i in range(block_size): 
    context = x[:i+1]
    target = y[i]
    print(f"when the input is {context} the target is: {target}")

when the input is tensor([18]) the target is: 47
when the input is tensor([18, 47]) the target is: 56
when the input is tensor([18, 47, 56]) the target is: 57
when the input is tensor([18, 47, 56, 57]) the target is: 58
when the input is tensor([18, 47, 56, 57, 58]) the target is: 1
when the input is tensor([18, 47, 56, 57, 58,  1]) the target is: 15
when the input is tensor([18, 47, 56, 57, 58,  1, 15]) the target is: 47
when the input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is: 58


In [8]:
torch.manual_seed(1337)

batch_size = 4 # number of independent sequences we will be processing in parallel
block_size = 8 # size of the chunks or maximum context length

def get_batches(split): 
    data = train_data if split == "train" else val_data
    ix = torch.randint((len(data) - block_size), (batch_size,)) # batch_size number of random integer from 0 to len(data) - block_size
    x = torch.stack([data[i:i+block_size] for i in ix]) # take the 1d tensors previously and stack them
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batches("train")
print(f"size of inputs: {xb.shape} \ninputs: {xb}")
print(f"size of ouputs: {yb.shape} \noutputs: {yb}")

print("---------")

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b][:t+1]
        target = yb[b][t]
        print(f"the context is: {context.tolist()} and the target is: {target}")

size of inputs: torch.Size([4, 8]) 
inputs: tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
size of ouputs: torch.Size([4, 8]) 
outputs: tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
---------
the context is: [24] and the target is: 43
the context is: [24, 43] and the target is: 58
the context is: [24, 43, 58] and the target is: 5
the context is: [24, 43, 58, 5] and the target is: 57
the context is: [24, 43, 58, 5, 57] and the target is: 1
the context is: [24, 43, 58, 5, 57, 1] and the target is: 46
the context is: [24, 43, 58, 5, 57, 1, 46] and the target is: 43
the context is: [24, 43, 58, 5, 57, 1, 46, 43] and the target is: 39
the context is: [44] and the target is: 53
the context is: [44, 53] and the target is: 56
the context is: [44, 53, 56

In [13]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module): 
    def __init__(self, vocab_size): 
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None): 
        # both idx and targets are of dimension (B, T) tensors
        logits = self.token_embedding_table(idx) #(B, T, C)

        if targets is None: 
            loss = None
        else:  
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens): 
        # idx is (B, T) array of indices in the current context 
        for _ in range(max_new_tokens): 
            logits, loss = self(idx) # get the predictions
            logits = logits[:, -1, :] # grabs the last element in the T (time) dimension, becomes (B, C)
            probs = F.softmax(logits, dim=-1) # (B, C)
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx
    
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(f"Shape of the logits tensor is: {logits.shape}")
print(f"Shape of loss tensor is: {loss.shape}, and the value is: {loss}")

idx = torch.zeros((1, 1), dtype=torch.long) # creating a (1, 1) tensor filled with 0 of type int
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist())) # need [0] to get individual batches of tensors

Shape of the logits tensor is: torch.Size([32, 65])
Shape of loss tensor is: torch.Size([]), and the value is: 4.878634929656982

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [14]:
# creating a PyTorch optimizer 
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [23]:
batch_size = 32

for step in range(10000): 
    xb, yb = get_batches("train")

    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True) 
    loss.backward()
    optimizer.step()

print(loss.item())

2.425787925720215


In [32]:
print(decode(m.generate(idx, max_new_tokens=400)[0].tolist()))



FORY thera che omariuty lousthende:
SIIUCETES: ofey senade l, fio esiel d


Sulss
Thowad.
OMERond, wne whod hand'lk ices! peye.
Hakn swe
An dar n ft'doo thede y bld he anoomlspe s; yapo or youg,
WI h s whal mitcovoumin my ciget k este ts teng fameanay-
wod.

Fotisth-pusunin wbed tet
CLBENI'suge shat, mpr'thof thotwil angn t tus th ne nshishe thidabjef;
UCE:
SSAn,
INTOfo fuseatathirtheriowenthice 
